# 25_26_MoS_Parameters
**Every number and data file used by the Method of Sections (MoS) simulation, in one place.** The other notebooks only *use* these values, so a design change is made here once and every notebook picks it up the next time `Test_Flight_MoS` is run.

**Conventions used throughout**
- **SI units:** metres, kilograms, seconds, newtons. Feet only appear for apogee targets and plots.
- **Positions:** measured from the **nose tip, positive toward the tail** (the rocket uses `coordinate_system_orientation="nose_to_tail"`). This matches how OpenRocket reports positions, so values can be copied straight from the `.ork` file.
- **Source:** values were taken from `Method_of_Sections.ork` unless a comment says otherwise.

## Imports
These libraries are imported here so every notebook that runs this one gets them as well.
- `rocketpy`: the 6-DOF flight simulator (`Environment`, `SolidMotor`, `Rocket`, `Flight`).
- `numpy` / `pandas`: array maths and reading the Excel data files.
- `datetime`: picks the launch date for the weather forecast.
- `matplotlib`: plotting.

In [ ]:
from rocketpy import Rocket, SolidMotor, Environment, Flight   # simulation classes
import numpy as np                                              # arrays and maths
import pandas as pd                                             # reading .xlsx data files
import datetime                                                 # launch date for the forecast
import matplotlib.pyplot as plt                                 # plotting

## Launch site, atmosphere and launch rail
Used by `Environment_IREC_Launch_Site` and by the `Flight` in `Test_Flight_MoS`.
- **Site:** Midland, TX (IREC launch site). `LAUNCH_ELEVATION` is a starting value; with the GFS forecast, RocketPy replaces it with the forecast model's ground height (it reported 851.2 m in your last run).
- **`ATMOSPHERE_MODEL`:**
  - `"Forecast"` pulls the real GFS weather forecast (wind, temperature and pressure at every altitude). It needs internet access and only works for dates within about 16 days.
  - `"standard_atmosphere"` is the no-wind ISA atmosphere. Use it for the stability plot IREC asks for (DTEG 10.2.3 requires a no-wind simulation).
- **Rail:** `INCLINATION` is measured from horizontal (90° = vertical) and `HEADING` from north. DTEG 10.1.1 says single-stage rockets are typically launched at **84° ±2°**.

In [ ]:
# ---- Launch site ---------------------------------------------------------------
LAUNCH_LAT = 31.99                 # deg N  (IREC launch site, Midland TX)
LAUNCH_LON = -102.07               # deg E
LAUNCH_ELEVATION = 890             # m above sea level (GFS overrides with its ground height)

# ---- Atmosphere / date --------------------------------------------------------
ATMOSPHERE_MODEL = "Forecast"      # "Forecast" (GFS weather) or "standard_atmosphere" (no wind)
LAUNCH_DATE = datetime.date.today() + datetime.timedelta(days=1)   # tomorrow's forecast
LAUNCH_HOUR_UTC = 12               # hour of the forecast to use, UTC

# ---- Launch rail ----------------------------------------------------------------
RAIL_LENGTH = 5.2                  # m  (IREC rail is 17 ft = 5.18 m, DTEG 5.3.1)
INCLINATION = 87                   # deg from horizontal (DTEG 10.1.1: typically 84 +/- 2)
HEADING = 0                        # deg from north

## Mass properties (rocket **without** motor)
RocketPy keeps the airframe and the motor separate: it adds the motor's changing mass, CG and inertia during the burn itself. These numbers must therefore exclude the motor.
- **`DRY_INERTIA` = (I₁₁, I₂₂, I₃₃)** in kg·m², taken about the dry CG: pitch, yaw, then roll. These were checked against OpenRocket. With the motor case added back, they reproduce OpenRocket's burnout inertia (13.46 kg·m²) exactly.
- **Check:** `DRY_MASS` plus the 8.064 kg motor gives 28.276 kg, which matches OpenRocket's launch mass.

In [ ]:
# ---- Mass properties (rocket WITHOUT motor) ----------------------------------
DRY_MASS = 20.212                    # kg
DRY_CG = 1.761                       # m from nose tip
DRY_INERTIA = (11.05, 11.05, 0.104)  # kg*m^2 about the dry CG: (pitch, yaw, roll)

# ---- Body --------------------------------------------------------------------
BODY_RADIUS = 0.078359               # m  (6.17 in OD); also sets RocketPy's reference area
ROCKET_LENGTH = 3.002915             # m  nose tip to aft end of boat tail

## Aerodynamic surfaces
RocketPy uses these surfaces for **lift and centre of pressure (CP)**, not drag. Each one gets a normal-force slope (CNα) and a CP location from Barrowman-type equations corrected for Mach. That gives the restoring moment and the stability margin. All drag comes from the RASAero file further down, so nothing is counted twice.
- **Nose:** OpenRocket's "Haack, shape parameter 0" is the same shape as RocketPy's `"vonKarman"`.
- **Fins:** `FIN_POSITION` is the **leading edge of the fin root**. The fins run past the start of the boat tail; RocketPy treats them as sitting on the full body radius, which is a small approximation.
- **Boat tail:** `TAIL_POSITION` is where the taper starts.

In [ ]:
# ---- Nose cone ---------------------------------------------------------------
NOSE_LENGTH = 0.78359                # m
NOSE_KIND = "vonKarman"              # = OpenRocket Haack, shape parameter 0
NOSE_POSITION = 0.0                  # m  nose tip is the origin

# ---- Fins (trapezoidal, x3) --------------------------------------------------
FIN_COUNT = 3
FIN_ROOT_CHORD = 0.193675            # m
FIN_TIP_CHORD = 0.0254               # m
FIN_SPAN = 0.2413                    # m  (semi-span, root to tip)
FIN_SWEEP_LENGTH = 0.168275          # m  leading-edge sweep, measured along the body axis
FIN_POSITION = 2.778125              # m  leading edge of the fin root
FIN_CANT = 0.0                       # deg  no cant -> no forced roll

# ---- Boat tail ---------------------------------------------------------------
TAIL_TOP_RADIUS = 0.078359           # m  matches the body radius
TAIL_BOTTOM_RADIUS = 0.057404        # m
TAIL_LENGTH = 0.1651                 # m
TAIL_POSITION = 2.837815             # m  start of the taper

## Rail buttons
RocketPy uses the button positions to work out the **effective rail length**: the distance the rocket travels before the upper (forward) button leaves the rail and it's free to rotate. That's the same definition IREC uses (DTEG 5.3.2), and the speed at that point must be **≥ 25 m/s** (DTEG 5.3.1).

The positions are approximate. Check them against CAD, because the upper button sets the rail exit velocity.

In [ ]:
# ---- Rail buttons (approximate) ----------------------------------------------
RAIL_BUTTON_UPPER = 1.733            # m from nose tip (forward button)
RAIL_BUTTON_LOWER = 2.273            # m from nose tip (aft button)
RAIL_BUTTON_ANGLE = 150.0            # deg around the body (matches the .ork)

## Motor: AeroTech M2500T
- **Thrust curve:** read from the `.eng` file. Only lines that start with a digit are kept (that skips the header and comment lines), and a `[0, 0]` point is added so thrust starts at zero.
  - **Duplicate times:** if the `.eng` already starts at t = 0, the extra row makes two identical times. Delete the `[0, 0]` row if so.
  - **The `invalid value encountered in divide` warning** printed during the flight doesn't come from this data. It's raised inside RocketPy when the flight ends and it converts the parachutes' pressure-signal data into functions, and it doesn't affect the results.
- **Grain geometry:** approximated, since the manufacturer doesn't publish it. RocketPy uses the grains to work out how the propellant's mass, CG and inertia change as it burns. `GRAIN_DENSITY` is computed so the four grains add up to exactly the published propellant mass.
- **Case inertia:** the empty case is modelled as a solid cylinder, I = m(3r² + L²)/12 in pitch and mr²/2 in roll. That's a good enough approximation at this size.
- **`MOTOR_POSITION`:** the nozzle exit, 12.7 mm aft of the airframe because of the motor overhang set in the `.ork`.

In [ ]:
# ---- Motor: AeroTech M2500T (98 mm, 751 mm long) ----------------------------
MOTOR_TOTAL_MASS = 8.064                                  # kg loaded
MOTOR_PROPELLANT_MASS = 4.659                             # kg
MOTOR_DRY_MASS = MOTOR_TOTAL_MASS - MOTOR_PROPELLANT_MASS  # kg empty case
MOTOR_LENGTH = 0.751                                      # m
MOTOR_RADIUS = 0.049                                      # m (98 mm diameter)
MOTOR_BURN_TIME = 4.264                                   # s
MOTOR_POSITION = ROCKET_LENGTH + 0.0127                   # m  nozzle exit (12.7 mm overhang)

# Thrust curve: keep only numeric (time, thrust) lines from the .eng file, add a (0, 0) start point
THRUST_CURVE = np.vstack([[0, 0], np.array([l.split() for l in open("AeroTech_M2500T.eng") if l.strip()[:1].isdigit()], float)])

# ---- Propellant grains (approximate geometry) --------------------------------
N_GRAINS = 4
GRAIN_R_OUT = 0.0400                 # m  grain outer radius
GRAIN_R_IN = 0.0160                  # m  initial core radius
GRAIN_H = 0.160                      # m  grain length
GRAIN_SEP = 0.004                    # m  gap between grains
GRAIN_VOL = np.pi * (GRAIN_R_OUT**2 - GRAIN_R_IN**2) * GRAIN_H * N_GRAINS   # m^3 total
GRAIN_DENSITY = MOTOR_PROPELLANT_MASS / GRAIN_VOL                           # kg/m^3, forces correct mass

# ---- Empty case inertia (solid-cylinder approximation) -----------------------
case_I_long = MOTOR_DRY_MASS * (3 * MOTOR_RADIUS**2 + MOTOR_LENGTH**2) / 12   # pitch/yaw
case_I_roll = MOTOR_DRY_MASS * MOTOR_RADIUS**2 / 2                            # roll

## Recovery
RocketPy models a parachute by its **drag area, CdS** (m²), not by its Cd and diameter separately, so `cd_s()` converts the parachute specs.

The main's CdS is the **sum of drogue, pilot and main**. When a new parachute deploys, RocketPy *replaces* the descent drag rather than adding to it, and on the real rocket all three are still out under the main.

`Test_Flight_MoS` stops at apogee (`terminate_on_apogee=True`), so these only matter once you simulate the descent.

In [ ]:
# ---- Recovery (from the .ork file) ------------------------------------------
def cd_s(cd, diameter):
    """Drag area CdS [m^2] from a parachute's Cd and nominal diameter [m]."""
    return cd * np.pi * diameter**2 / 4

DROGUE_CDS = cd_s(1.55, 0.9144)   # Fruity Chutes CFC-036, 36 in elliptical
PILOT_CDS  = cd_s(0.97, 0.6096)   # Rocketman LS-02, 24 in pilot
MAIN_CDS   = cd_s(2.20, 1.8288)   # Fruity Chutes IFC-072, 72 in Iris Ultra
MAIN_DEPLOY_ALT = 457.2           # m AGL (1500 ft); RocketPy treats a number trigger as height above ground

## Drag data
- **`CD` (rocket drag):** Mach against Cd from RASAero II, read from the first two columns. RocketPy takes it as a list of `[mach, cd]` pairs. The Cd must be referenced to the body cross-section π·`BODY_RADIUS`², which is RocketPy's reference area. The same curve is used for power-on and power-off; RASAero's separate power-on column (lower base drag under thrust) would be slightly more accurate during the burn.
- **`AFC_Drag_Data` (air brakes):** a 3-column table of **deployment level (0–1), Mach, Cd**. Row 1 of the sheet is treated as a header. Because the rocket is set up with `override_rocket_drag=False`, this Cd is **added** to the rocket's Cd, so the table should hold the extra drag the brakes cause, referenced to the same body area.
- **`CD_ROCKET`:** the same rocket Cd as a NumPy array, so the controller can interpolate it quickly.

In [ ]:
# ---- Cd data from RASAero II ------------------------------------------------
CD = pd.read_excel("RASAeroII Cd Data.xlsx", header=None, usecols=[0, 1]).to_numpy(float).tolist()   # [[mach, cd], ...]
CD_ROCKET = np.array(CD)                                                                          # same data, array form

# ---- Air-brake (AFC) drag table: columns = deployment level, Mach, Cd ---------
AFC_Drag_Data = pd.read_excel("airbrake_cd.xlsx").to_numpy(float)

## Air-brake controller settings
Used by `controller_function` in `Method_of_Sections`.
- **`TARGET_APOGEE_FT`:** the apogee the brakes aim for. IREC scores apogee relative to 10,000 ft AGL.
- **`MAX_RATE`:** the fastest the brakes can move, as a fraction of full travel per second (0.5 means fully open in 2 s). Set it to what your actuator can actually do.
- **`AIRBRAKE_SAMPLING_RATE`:** how often the controller runs, in Hz, like a flight computer's loop rate.
- **`G`:** standard gravity, used in the controller's apogee prediction.

In [ ]:
# ---- Air-brake controller ------------------------------------------------------
TARGET_APOGEE_FT = 10000          # ft AGL
MAX_RATE = 0.5                    # max deployment change per second (fraction of full travel)
AIRBRAKE_SAMPLING_RATE = 10       # Hz, controller update rate
G = 9.80665                       # m/s^2

## Stability requirements and unit helpers
These are the limits drawn on the stability plot in `Test_Flight_MoS`. The IREC minimum (7.5% subsonic or 10% supersonic, DTEG 10.2.1) is chosen automatically there from the flight's peak Mach.

In [ ]:
# ---- Stability requirements (IREC 2026 DTEG + team) ---------------------------
MAX_LAUNCH_PCT = 18.0             # % of rocket length, max at launch      (DTEG 10.3.1.1)
MAX_FLIGHT_PCT = 25.0             # % of rocket length, max through flight (DTEG 10.3.1.2)
TEAM_MIN_CAL   = 1.5              # calibers, MARS design requirement

# ---- Unit helpers ---------------------------------------------------------------
FT = 3.28084                      # m -> ft
DIAMETER = 2 * BODY_RADIUS        # m, one caliber